# Planes, segments, boxes, rays

Curb faces are often vertical. A fit of the form `z = ax + by + c` cannot represent them. Use a general plane, and keep a tolerance instead of comparing floats with `==`.


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)


## 1. Fit a plane and measure distance

Fit one plane to every point. Return the centroid and a unit normal. Also write the signed distance from arbitrary points to that plane, positive on the normal's side.

Use a general plane `ax + by + cz = d`. Flip the normal so its largest component is positive; the SVD sign is arbitrary, and a stable sign makes tests possible.

The fit uses every point. One outlier will tilt it. That is what the RANSAC problem is for.

Hand check: points with `x = 0` and varying `y, z` have normal `(1, 0, 0)`, not a normal with a large `z` component.

<details><summary>Hint</summary>

Center the points. `_, _, Vt = np.linalg.svd(centered, full_matrices=False)`. The normal is `Vt[-1]`, the direction of least variance. Signed distance is `(points - centroid) @ unit_normal`.

</details>


In [ ]:
import numpy as np

def fit_plane(points):
    """
    Least-squares plane through all points.

    Returns centroid (3,) and a unit normal (3,).
    The normal's largest component is positive.
    Fit a general plane, including vertical faces.
    """
    raise NotImplementedError

def signed_plane_distances(points, centroid, normal):
    """
    Signed distance from each point to the plane. Positive along `normal`.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def fit_plane(points):
    """Least-squares plane through all points.

    Returns centroid (3,) and a unit normal (3,).
    The normal's largest component is positive.
    Fit a general plane, including vertical faces.
    """
    points = np.asarray(points, dtype=float)
    if len(points) < 3:
        raise ValueError("need at least 3 points")
    centroid = points.mean(axis=0)
    # Vt rows are principal axes, last row = least variance = plane normal.
    # A height field z = ax + by + c cannot represent a vertical curb face.
    _, _, Vt = np.linalg.svd(points - centroid, full_matrices=False)
    normal = Vt[-1].copy()
    # Sign is arbitrary. Make the largest component positive so the answer is stable.
    k = int(np.argmax(np.abs(normal)))
    if normal[k] < 0.0:
        normal = -normal
    return centroid, normal

def signed_plane_distances(points, centroid, normal):
    """Signed distance from each point to the plane. Positive along `normal`."""
    points = np.asarray(points, dtype=float)
    normal = np.asarray(normal, dtype=float)
    n = np.linalg.norm(normal)
    if n < 1e-12:
        raise ValueError("normal has zero length")
    unit = normal / n
    return (points - np.asarray(centroid, dtype=float)) @ unit

def check_fit_plane():
    rng = np.random.default_rng(0)
    xy = rng.normal(size=(30, 2))
    horizontal = np.column_stack([xy, 0.01 * rng.normal(size=30)])
    c, n = fit_plane(horizontal)
    assert np.allclose(n, [0, 0, 1], atol=0.05)
    assert np.allclose(np.abs(signed_plane_distances(horizontal, c, n)).max(), 0.03, atol=0.02)
    vertical = np.column_stack([np.zeros(20), rng.normal(size=20), rng.normal(size=20)])
    _, n = fit_plane(vertical)
    assert np.allclose(np.abs(n), [1, 0, 0], atol=1e-8)
    assert n[0] > 0
    print("passed")

check_fit_plane()


## 2. Point-to-segment distance

For many points and one finite segment `ab`, return the Euclidean distance to the segment and the closest point on the segment.

Clamp the segment parameter to `[0, 1]`. If you leave it unclamped, you have distance to the infinite line, which is a different question and worth naming if they ask. A zero-length segment is just distance to `a`.

Hand check: segment `(0,0,0)`–`(1,0,0)`. The point `(0.5, 1, 0)` is at distance 1, closest `(0.5, 0, 0)`. The point `(2, 0, 0)` is at distance 1, closest `(1, 0, 0)`.


In [ ]:
import numpy as np

def point_to_segment(points, a, b):
    """
    Distance from each point to the finite segment ab.

    points: (N, D), a and b: (D,)
    returns distances (N,), closest points (N, D).
    `t` is clamped to [0, 1]. Leave it unclamped for an infinite line.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def point_to_segment(points, a, b):
    """Distance from each point to the finite segment ab.

    points: (N, D), a and b: (D,)
    returns distances (N,), closest points (N, D).
    `t` is clamped to [0, 1]. Leave it unclamped for an infinite line.
    """
    points = np.asarray(points, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ab = b - a
    denom = float(ab @ ab)
    if denom < 1e-24:
        closest = np.broadcast_to(a, points.shape).copy()
        return np.linalg.norm(points - a, axis=1), closest
    t = ((points - a) @ ab) / denom
    t = np.clip(t, 0.0, 1.0)
    closest = a + t[:, None] * ab
    return np.linalg.norm(points - closest, axis=1), closest

def check_point_to_segment():
    pts = np.array([[0.5, 1.0, 0.0], [0.0, 1.0, 0.0], [2.0, 0.0, 0.0], [0.2, 0.2, 0.0]])
    dist, closest = point_to_segment(pts, np.array([0.0, 0.0, 0.0]), np.array([1.0, 0.0, 0.0]))
    assert np.allclose(dist, [1.0, 1.0, 1.0, 0.2])
    assert np.allclose(closest[0], [0.5, 0.0, 0.0])
    assert np.allclose(closest[2], [1.0, 0.0, 0.0])
    dist, closest = point_to_segment(pts[:1], np.array([1.0, 1.0, 1.0]), np.array([1.0, 1.0, 1.0]))
    assert np.allclose(dist, [np.linalg.norm(pts[0] - 1)])
    print("passed")

check_point_to_segment()


## 3. 3D axis-aligned box IoU

Intersection over union of two axis-aligned boxes. Each box is a min corner and a max corner, shape `(3,)`.

Intersect by clipping each axis independently, then multiply the three positive extents. A shared face has volume 0, so IoU 0.

Hand check: `[0,0,0]–[2,2,2]` against `[1,1,1]–[3,3,3]` has intersection volume 1 and union 15, so IoU `1/15`.


In [ ]:
import numpy as np

def aabb_iou(a_min, a_max, b_min, b_max):
    """
    Intersection-over-union of two axis-aligned boxes.

    Each argument is (D,). Works in 2D or 3D. Touching faces have IoU 0
    because the intersection volume is zero.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def aabb_iou(a_min, a_max, b_min, b_max):
    """Intersection-over-union of two axis-aligned boxes.

    Each argument is (D,). Works in 2D or 3D. Touching faces have IoU 0
    because the intersection volume is zero.
    """
    a_min = np.asarray(a_min, dtype=float)
    a_max = np.asarray(a_max, dtype=float)
    b_min = np.asarray(b_min, dtype=float)
    b_max = np.asarray(b_max, dtype=float)
    inter_min = np.maximum(a_min, b_min)
    inter_max = np.minimum(a_max, b_max)
    inter_ext = np.maximum(inter_max - inter_min, 0.0)
    inter = float(np.prod(inter_ext))
    vol_a = float(np.prod(a_max - a_min))
    vol_b = float(np.prod(b_max - b_min))
    union = vol_a + vol_b - inter
    if union <= 0.0:
        return 0.0
    return inter / union

def check_aabb_iou():
    iou = aabb_iou([0, 0, 0], [2, 2, 2], [1, 1, 1], [3, 3, 3])
    assert np.allclose(iou, 1.0 / 15.0)
    assert aabb_iou([0, 0, 0], [1, 1, 1], [0, 0, 0], [1, 1, 1]) == 1.0
    assert aabb_iou([0, 0, 0], [1, 1, 1], [2, 2, 2], [3, 3, 3]) == 0.0
    assert aabb_iou([0, 0, 0], [1, 1, 1], [1, 1, 1], [2, 2, 2]) == 0.0
    print("passed")

check_aabb_iou()


## 4. Points inside an oriented box

An oriented box is a center, a rotation, and per-axis half extents. `R` maps local coordinates to world coordinates: its columns are the box axes in the world. A world point `p` has local coordinates `R.T @ (p - center)`.

Return a boolean mask. Treat the boundary as inside, with a small tolerance so a point on a face is not rejected by rounding.

Hand check: `R` is a `+90°` rotation about Z,

```
[[0, -1, 0],
 [1,  0, 0],
 [0,  0, 1]]
```

half extents `(2, 0.5, 1)`, center at the origin. The world point `(0, 1.5, 0)` is inside. The world point `(1.5, 0, 0)` is outside. Getting those swapped means the local transform is transposed.

<details><summary>Hint</summary>

For row points, `local = (points - center) @ R`.

</details>


In [ ]:
import numpy as np

def points_in_obb(points, center, R, half_extents, tol=1e-08):
    """
    Which points lie in an oriented box.

    World point p corresponds to local = R.T @ (p - center), and R's columns
    are the box axes expressed in world coordinates. A point is inside when
    every |local_i| <= half_extents[i].
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def points_in_obb(points, center, R, half_extents, tol=1e-8):
    """Which points lie in an oriented box.

    World point p corresponds to local = R.T @ (p - center), and R's columns
    are the box axes expressed in world coordinates. A point is inside when
    every |local_i| <= half_extents[i].
    """
    points = np.asarray(points, dtype=float)
    center = np.asarray(center, dtype=float)
    half_extents = np.asarray(half_extents, dtype=float)
    local = (points - center) @ R
    return np.all(np.abs(local) <= half_extents + tol, axis=1)

def check_points_in_obb():
    R = np.array([[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]])
    pts = np.array([[0.0, 1.5, 0.0], [1.5, 0.0, 0.0], [0.0, 0.0, 0.9], [0.0, 0.0, 1.2]])
    mask = points_in_obb(pts, np.zeros(3), R, np.array([2.0, 0.5, 1.0]))
    assert np.array_equal(mask, [True, False, True, False])
    print("passed")

check_points_in_obb()


## 5. Ray-triangle intersection

Intersect many rays with one triangle using Möller–Trumbore. `directions` are not necessarily unit length; `t` is the scale in `origin + t * direction`.

Count a hit when `t > 0` and the barycentric coordinates lie inside the triangle, including the boundary. A parallel ray misses. A ray that starts on the triangle and points away misses. `t` only needs to be meaningful where the hit mask is true.

Hand check: triangle `(0,0,0)`, `(1,0,0)`, `(0,1,0)`.

- origin `(0.25, 0.25, 1)`, direction `(0, 0, -1)` hits at `t = 1`
- origin `(0, 0, 2)`, direction `(0, 0, -1)` hits the vertex, `t = 2`
- origin `(0.5, 0.5, 1)`, direction `(0, 0, -1)` hits the hypotenuse
- origin `(2, 2, 1)` misses
- origin `(0.25, 0.25, -1)`, direction `(0, 0, -1)` misses (`t` would be negative)
- a ray lying in the plane misses

<details><summary>Hint</summary>

`h = dir × edge2`, `a = edge1 · h`. Skip `|a| < eps`. Then `u`, `v`, and `t` are the usual three scaled dots. Guard the division so a parallel ray does not warn.

</details>


In [ ]:
import numpy as np

def ray_triangle(origins, directions, v0, v1, v2, eps=1e-08):
    """
    Möller–Trumbore intersection of many rays with one triangle.

    origins, directions: (N, 3). `t` solves origin + t * direction, and the
    direction does not need to be a unit vector.
    A hit requires t > eps (in front of the origin) and barycentric coordinates
    inside the triangle, including the boundary.
    Parallel rays miss. `t` is only meaningful where hit is True.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def ray_triangle(origins, directions, v0, v1, v2, eps=1e-8):
    """Möller–Trumbore intersection of many rays with one triangle.

    origins, directions: (N, 3). `t` solves origin + t * direction, and the
    direction does not need to be a unit vector.
    A hit requires t > eps (in front of the origin) and barycentric coordinates
    inside the triangle, including the boundary.
    Parallel rays miss. `t` is only meaningful where hit is True.
    """
    origins = np.asarray(origins, dtype=float)
    directions = np.asarray(directions, dtype=float)
    v0 = np.asarray(v0, dtype=float)
    v1 = np.asarray(v1, dtype=float)
    v2 = np.asarray(v2, dtype=float)
    edge1 = v1 - v0
    edge2 = v2 - v0
    h = np.cross(directions, edge2)
    a = h @ edge1
    parallel = np.abs(a) < eps
    f = np.divide(1.0, a, out=np.zeros_like(a), where=~parallel)
    s = origins - v0
    u = f * np.sum(s * h, axis=1)
    q = np.cross(s, edge1)
    v = f * np.sum(directions * q, axis=1)
    t = f * (q @ edge2)
    hit = (~parallel) & (u >= 0.0) & (v >= 0.0) & (u + v <= 1.0) & (t > eps)
    return t, hit

def check_ray_triangle():
    v0, v1, v2 = np.eye(3)[0] * 0, np.array([1.0, 0.0, 0.0]), np.array([0.0, 1.0, 0.0])
    v0 = np.zeros(3)
    origins = np.array(
        [
            [0.25, 0.25, 1.0],
            [0.0, 0.0, 2.0],
            [0.5, 0.5, 1.0],
            [2.0, 2.0, 1.0],
            [0.25, 0.25, -1.0],
            [0.2, 0.2, 0.0],
        ]
    )
    directions = np.array(
        [
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [0.0, 0.0, -1.0],
            [1.0, 0.0, 0.0],
        ]
    )
    t, hit = ray_triangle(origins, directions, v0, v1, v2)
    assert np.array_equal(hit, [True, True, True, False, False, False])
    assert np.allclose(t[0], 1.0)
    assert np.allclose(origins[0] + t[0] * directions[0], [0.25, 0.25, 0.0])
    assert np.allclose(t[1], 2.0)
    print("passed")

check_ray_triangle()


## 6. RANSAC plane

Fit a plane when some points do not belong to it. Each hypothesis is 3 points. Skip a collinear sample. Score a hypothesis by how many points fall within `threshold` of its plane. After the loop, refit a least-squares plane on the inliers and recompute the mask from that refit plane.

A Python loop over hypotheses is the right shape. The distance test inside a hypothesis should be vectorized. Use `np.random.default_rng(seed)` so a given seed is reproducible.

The normal follows the same sign rule as the least-squares fit: largest component positive.

Hand check: a set of points on `z = 0` plus a few points at `z ≈ 5`, threshold `0.05`. The normal is `(0,0,1)` and the high points are outliers. Repeat the idea on the plane `x = 0` with outliers at `x = 5`. A vertical inlier set is the one that fails if you parameterized the plane as a height field.

The reference cell includes the least-squares helpers so it runs alone.


In [ ]:
import numpy as np

def ransac_plane(points, threshold, iterations, seed=0):
    """
    Fit a plane by RANSAC, then refit the plane on the inliers.

    Returns centroid, unit normal (largest component positive), and a boolean
    inlier mask measured against the refit plane.
    Hypotheses that are collinear are skipped. A Python loop over iterations
    is the expected shape; each trial itself is vectorized over the points.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

# Included so this cell runs by itself, including helpers from earlier problems.

def fit_plane(points):
    """Least-squares plane through all points.

    Returns centroid (3,) and a unit normal (3,).
    The normal's largest component is positive.
    Fit a general plane, including vertical faces.
    """
    points = np.asarray(points, dtype=float)
    if len(points) < 3:
        raise ValueError("need at least 3 points")
    centroid = points.mean(axis=0)
    # Vt rows are principal axes, last row = least variance = plane normal.
    # A height field z = ax + by + c cannot represent a vertical curb face.
    _, _, Vt = np.linalg.svd(points - centroid, full_matrices=False)
    normal = Vt[-1].copy()
    # Sign is arbitrary. Make the largest component positive so the answer is stable.
    k = int(np.argmax(np.abs(normal)))
    if normal[k] < 0.0:
        normal = -normal
    return centroid, normal

def signed_plane_distances(points, centroid, normal):
    """Signed distance from each point to the plane. Positive along `normal`."""
    points = np.asarray(points, dtype=float)
    normal = np.asarray(normal, dtype=float)
    n = np.linalg.norm(normal)
    if n < 1e-12:
        raise ValueError("normal has zero length")
    unit = normal / n
    return (points - np.asarray(centroid, dtype=float)) @ unit

def ransac_plane(points, threshold, iterations, seed=0):
    """Fit a plane by RANSAC, then refit the plane on the inliers.

    Returns centroid, unit normal (largest component positive), and a boolean
    inlier mask measured against the refit plane.
    Hypotheses that are collinear are skipped. A Python loop over iterations
    is the expected shape; each trial itself is vectorized over the points.
    """
    points = np.asarray(points, dtype=float)
    if len(points) < 3:
        raise ValueError("need at least 3 points")
    rng = np.random.default_rng(seed)
    best_inliers = None
    best_count = -1
    n = len(points)
    for _ in range(iterations):
        choice = rng.choice(n, size=3, replace=False)
        a, b, c = points[choice]
        normal = np.cross(b - a, c - a)
        length = np.linalg.norm(normal)
        if length < 1e-12:
            continue
        normal = normal / length
        dist = np.abs((points - a) @ normal)
        inliers = dist <= threshold
        count = int(inliers.sum())
        if count > best_count:
            best_count = count
            best_inliers = inliers
    if best_inliers is None or best_count < 3:
        raise RuntimeError("no valid plane hypothesis")
    centroid, normal = fit_plane(points[best_inliers])
    inliers = np.abs(signed_plane_distances(points, centroid, normal)) <= threshold
    # Refit once more in case the threshold pulled in a few extra points.
    if int(inliers.sum()) >= 3:
        centroid, normal = fit_plane(points[inliers])
        inliers = np.abs(signed_plane_distances(points, centroid, normal)) <= threshold
    return centroid, normal, inliers

def check_ransac_plane():
    ground = np.column_stack([np.linspace(-1, 1, 20), np.linspace(-1, 1, 20), np.zeros(20)])
    extra = np.array([[0.2, -0.4, 0.0], [-0.7, 0.3, 0.0], [0.5, 0.5, 0.0]])
    outliers = np.array([[0.0, 0.0, 5.0], [0.4, -0.2, 4.0], [-0.3, 0.6, 6.0]])
    pts = np.vstack([ground, extra, outliers])
    _, normal, mask = ransac_plane(pts, threshold=0.05, iterations=40, seed=0)
    assert np.allclose(normal, [0, 0, 1], atol=1e-6)
    assert mask.sum() == len(pts) - len(outliers)
    assert not mask[-len(outliers) :].any()

    rng = np.random.default_rng(1)
    y = rng.uniform(-2, 2, 30)
    z = rng.uniform(-2, 2, 30)
    wall = np.column_stack([np.zeros(30), y, z])
    bad = rng.uniform(-2, 2, size=(10, 3))
    bad[:, 0] = 5.0
    pts = np.vstack([wall, bad])
    _, normal, mask = ransac_plane(pts, threshold=0.05, iterations=80, seed=1)
    assert np.allclose(normal, [1, 0, 0], atol=1e-6)
    assert mask.sum() == 30
    print("passed")

check_ransac_plane()
